# 04b — Costruzione Feature Matrix

Produce `data/feature_matrix_graph_v1.parquet`:
- 9096 righe (agenti nel grafo conversazionale)
- 20 colonne: `agent_id` + 19 feature ML
- Zero NaN
- Feature skewed trasformate con log1p

**Prerequisiti**: 04a eseguito senza BUG.

## 1. Load dal DB

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../data/moltbook.db")

df_raw = pd.read_sql("SELECT * FROM agent_features", conn)
conn.close()

print(f"Righe totali: {len(df_raw)}")
print(f"Colonne: {list(df_raw.columns)}")

## 2. Filtro subset analitico

Teniamo solo gli agenti nel grafo conversazionale (in_degree valorizzato).

In [ ]:
df_graph = df_raw[df_raw["in_degree"].notna()].copy()

print(f"Agenti nel grafo: {len(df_graph)}")
assert len(df_graph) >= 9000, f"Attesi almeno 9000 agenti, trovati {len(df_graph)} — verificare dataset"
print(f"Shape: {df_graph.shape}")

## 3. Selezione ML features

19 feature selezionate. Escluse:
- `unique_targets`: correlazione ~1.0 con `out_degree`
- `egonet_size`: correlazione ~1.0 con `out_degree`
- `is_claimed`: 98.9% claimed → sbilanciamento estremo, non utile per clustering
- Colonne di metadati: `computed_at`, `feature_version`, `first_activity`, `last_activity`

In [ ]:
ML_FEATURES = [
    # Attività
    "n_posts", "n_comments", "n_comments_received", "active_days",
    # Temporali
    "burstiness_posts", "hour_entropy",
    # Comportamentali
    "reply_to_post_ratio", "self_reply_rate", "mean_thread_depth",
    # Testuali
    "mean_post_length", "std_post_length", "type_token_ratio",
    # Rete
    "in_degree", "out_degree", "pagerank", "betweenness",
    "local_clustering", "egonet_density", "reciprocity_local",
]
# NOTA: unique_targets e egonet_size esclusi per ridondanza (corr > 0.9999 con out_degree)

df_ml = df_graph[["agent_id"] + ML_FEATURES].copy()

print(f"Shape dopo selezione: {df_ml.shape}")
print(f"\nNaN per feature (prima dell'imputazione):")
nan_before = df_ml[ML_FEATURES].isnull().sum()
print(nan_before[nan_before > 0].to_string())

## 4. Policy NaN — imputazione con 0

Tutti i NaN nelle feature ML sono strutturalmente giustificati (verificato in 04a).
L'imputazione con 0 è semanticamente corretta per ogni caso:

| Feature | Causa NaN | Imputazione con 0 significa |
|---|---|---|
| `burstiness_posts` | n_posts < 3 | nessuna burstiness misurabile |
| `self_reply_rate` | nessun commento con parent_id | non si è mai risposto a se stesso |
| `mean_post_length` | n_posts = 0 | agente senza post |
| `std_post_length` | n_posts = 0 | agente senza post |
| `mean_thread_depth` | depth NULL nel DB | profondità non misurabile |
| `reply_to_post_ratio` | n_posts=0 e n_comments=0 | nessuna attività |
| `type_token_ratio` | nessun testo | vocabolario non misurabile |

In [ ]:
df_ml[ML_FEATURES] = df_ml[ML_FEATURES].fillna(0)

nan_after = df_ml[ML_FEATURES].isnull().sum().sum()
print(f"NaN totali dopo imputazione: {nan_after}")
assert nan_after == 0, "Ancora NaN dopo fillna — verificare"
print("OK — zero NaN")

## 5. Log-transform (log1p) sulle feature skewed

Trasformiamo le feature con distribuzione fortemente skewed.
`log1p(x) = log(1+x)` è sicuro anche per x=0.

NON trasformiamo:
- Feature già bounded in [0,1]: `hour_entropy`, `type_token_ratio`, `local_clustering`,
  `self_reply_rate`, `egonet_density`, `reciprocity_local`, `reply_to_post_ratio`
- Feature che possono essere negative: `burstiness_posts` (range [-1, 1])
- Feature ordinali/conteggi piccoli: `active_days`, `mean_thread_depth`
- Feature di lunghezza testo: `mean_post_length`, `std_post_length`

In [ ]:
SKEWED_FEATURES = [
    "n_posts", "n_comments", "n_comments_received",
    "in_degree", "out_degree", "pagerank", "betweenness",
    "reply_to_post_ratio",
]

# Verifica che tutte le feature skewed siano >= 0 (log1p richiede x >= 0)
for col in SKEWED_FEATURES:
    min_val = df_ml[col].min()
    if min_val < 0:
        print(f"ATTENZIONE: {col} ha valori negativi (min={min_val:.4f}) — skip log1p")
    else:
        df_ml[col] = np.log1p(df_ml[col])

print("Log1p applicato a:", SKEWED_FEATURES)
print("\nRange post-trasformazione:")
print(df_ml[SKEWED_FEATURES].agg(["min", "max", "mean"]).round(4).to_string())

## 6. Verifiche finali

In [ ]:
# Zero NaN
assert df_ml.isnull().sum().sum() == 0, "NaN residui nella feature matrix!"
print("NaN check: OK")

# Shape attesa
expected_cols = len(ML_FEATURES) + 1  # +1 per agent_id
assert df_ml.shape[1] == expected_cols, f"Colonne attese: {expected_cols}, trovate: {df_ml.shape[1]}"
print(f"Shape check: {df_ml.shape} — OK")

# Nessun valore infinito
inf_count = np.isinf(df_ml[ML_FEATURES]).sum().sum()
assert inf_count == 0, f"Trovati {inf_count} valori infiniti!"
print("Inf check: OK")

print("\nStatistiche descrittive post-trasformazione:")
print(df_ml[ML_FEATURES].describe().round(4).to_string())

In [ ]:
print("\nHead (5 righe):")
print(df_ml.head().to_string())

## 7. Salvataggio

In [ ]:
output_path = "../data/feature_matrix_graph_v1.parquet"
df_ml.to_parquet(output_path, index=False)

print(f"Salvato: {output_path}")
print(f"Shape: {df_ml.shape}")
print(f"Colonne: {list(df_ml.columns)}")

# Verifica idempotenza: rileggi e controlla shape
df_check = pd.read_parquet(output_path)
assert df_check.shape == df_ml.shape, "Shape mismatch al ricaricamento!"
print("\nVerifica ricaricamento: OK")